In [ ]:
import numpy as np
import pandas as pd
from pandas import DataFrame
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Libraries for retrieving data from the web and handling zip files
import requests, zipfile
from io import StringIO
import io

# Specify the url with data
url = 'https://github.com/Hernan4444/MyAnimeList-Database/archive/refs/heads/master.zip'

# Acquire data from the url
r = requests.get(url, stream=True)

# read and extract the zipfile
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall()

# Load and clean the data
anime_data = pd.read_csv('MyAnimeList-Database-master/data/anime.csv')
anime_data_extracted = anime_data[anime_data['Score'] != 'Unknown'].copy()
anime_data_extracted['Score'] = pd.to_numeric(anime_data_extracted['Score'])

In [ ]:
anime_data_extracted.head()

,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,...,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,...,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",Unknown,...,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,...,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,...,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,...,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0


In [ ]:


# Don't forget to erase the "!!WRITE ME!!" before submitting!
def homework(anime_data_extracted):
  from sklearn.preprocessing import LabelEncoder
  from sklearn.preprocessing import StandardScaler
  # Extract target variables
  x = anime_data_extracted[['Score']].copy() # Use .copy() to avoid SettingWithCopyWarning
  ss = StandardScaler()
  x_std_array = ss.fit_transform(x) # Renamed to avoid confusion with the x DataFrame

  # Convert the numpy array back to a DataFrame with original index and column name
  x_std_df = pd.DataFrame(x_std_array, columns=['Score_std'], index=x.index)

  # Compare pre- and post-transformation values
  # Concatenate original 'Score' and 'Score_std' for display
  display_df = pd.concat([x, x_std_df], axis=1)
  display(display_df)

  # Extract the 'Type' column for one-hot encoding.
  y = anime_data_extracted['Type']



  # Applying label Encoding (not directly used in `result` for concat, but kept if needed later)
  le = LabelEncoder()
  encoded = le.fit_transform(y)

  x_ohe = pd.get_dummies(y)
  x_ohe = x_ohe.astype(int)

  # Concatenate the standardized score DataFrame and the one-hot encoded DataFrame
  # Ensure both DataFrames have the same index for correct alignment during 'inner' join
  result = pd.concat([x_std_df, x_ohe], axis=1, join="inner")

  return result

In [ ]:
homework(anime_data_extracted)
#

,Score,Score_std
0,8.78,2.560109
1,8.39,2.120266
2,8.24,1.951096
3,7.27,0.857129
4,6.98,0.530067
...,...,...
17504,6.59,0.090225
17505,7.52,1.139080
17512,6.83,0.360897
17513,4.81,-1.917260


,Score_std,Movie,Music,ONA,OVA,Special,TV
0,2.560109,0,0,0,0,0,1
1,2.120266,1,0,0,0,0,0
2,1.951096,0,0,0,0,0,1
3,0.857129,0,0,0,0,0,1
4,0.530067,0,0,0,0,0,1
...,...,...,...,...,...,...,...
17504,0.090225,0,0,1,0,0,0
17505,1.139080,0,1,0,0,0,0
17512,0.360897,0,0,0,0,1,0
17513,-1.917260,0,0,0,0,1,0


In [ ]:
def homework(anime_data_extracted: pd.DataFrame) -> pd.DataFrame:
    # 1. Standardize Score column
    mean = anime_data_extracted['Score'].mean()
    std = anime_data_extracted['Score'].std()
    Score_std = (anime_data_extracted['Score'] - mean) / std

    # 2. One-Hot Encode Type column, cast to int
    type_dummies = pd.get_dummies(anime_data_extracted['Type'], dtype=int)

    # Ensure all expected columns are present (in order)
    for col in ['Movie', 'Music', 'ONA', 'OVA', 'Special', 'TV']:
        if col not in type_dummies.columns:
            type_dummies[col] = 0
    type_dummies = type_dummies[['Movie', 'Music', 'ONA', 'OVA', 'Special', 'TV']]

    # 3. Combine and return
    result = pd.concat([Score_std.rename('Score_std'), type_dummies], axis=1)
    return result

In [ ]:
homework(anime_data_extracted)

,Score_std,Movie,Music,ONA,OVA,Special,TV
0,2.560006,0,0,0,0,0,1
1,2.120181,1,0,0,0,0,0
2,1.951018,0,0,0,0,0,1
3,0.857095,0,0,0,0,0,1
4,0.530046,0,0,0,0,0,1
...,...,...,...,...,...,...,...
17504,0.090221,0,0,1,0,0,0
17505,1.139034,0,1,0,0,0,0
17512,0.360883,0,0,0,0,1,0
17513,-1.917183,0,0,0,0,1,0
